In [1]:
# Installer les bibliothèques nécessaires
!pip install requests geojson folium matplotlib pillow reportlab pypdf selenium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.0 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/11.8 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 91.4 MB/s  0:00:00


   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/11 [reportlab]

   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/11 [reportlab]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [selenium]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [selenium]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [selenium]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [selenium]



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [6]:
import requests
import json
import folium
import matplotlib.pyplot as plt
from PIL import Image
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage
from reportlab.lib.enums import TA_CENTER, TA_LEFT
import io
import os
from pathlib import Path

# Configuration
UMAP_ID = "1177975"  # ⚠️ À REMPLACER PAR VOTRE ID
UMAP_GEOJSON_URL = f"https://umap.openstreetmap.fr/fr/map/{UMAP_ID}/geojson"
OUTPUT_PDF = "mon_guide_voyage.pdf"
TITRE_GUIDE = "Mon Guide de Voyage"
DESCRIPTION = "Guide interactif généré automatiquement depuis Umap"

print("✓ Configuration chargée")
print(f"  ID Umap : {UMAP_ID}")
print(f"  URL API : {UMAP_GEOJSON_URL}")


✓ Configuration chargée
  ID Umap : 1177975
  URL API : https://umap.openstreetmap.fr/fr/map/1177975/geojson


In [9]:
def recuperer_donnees_umap(url):
    """Télécharge les données GeoJSON depuis Umap"""
    try:
        print(f"📥 Téléchargement depuis : {url}")
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        print(f"✓ Réponse reçue")
        print(f"📋 Clés principales : {list(data.keys())}")
        
        # Vérifier la structure
        if 'datalayers' in data:
            print(f"📊 {len(data['datalayers'])} couche(s) trouvée(s)")
            for i, layer in enumerate(data['datalayers']):
                layer_name = layer.get('properties', {}).get('name', f'Couche {i}')
                print(f"   - {layer_name}")
        
        return data
    except requests.exceptions.RequestException as e:
        print(f"❌ Erreur lors du téléchargement : {e}")
        return None

# Test
geojson = recuperer_donnees_umap(UMAP_GEOJSON_URL)
if geojson:
    print("\n✅ Données chargées avec succès")


📥 Téléchargement depuis : https://umap.openstreetmap.fr/fr/map/1177975/geojson
✓ Réponse reçue
📋 Clés principales : ['type', 'geometry', 'properties']

✅ Données chargées avec succès


In [12]:
def generer_carte_folium(umap_data, output_html="carte_temp.html", 
                         centre_lat=None, centre_lon=None, zoom=None):
    """
    Crée une carte Folium à partir des données Umap
    """
    print("🎨 Génération de la carte Folium...")
    
    # Récupérer le centre et le zoom depuis les métadonnées Umap
    if centre_lat is None:
        centre_lat = umap_data.get('center', [48.8566, 2.3522])[0]
        centre_lon = umap_data.get('center', [48.8566, 2.3522])[1]
    if zoom is None:
        zoom = umap_data.get('zoom', 12)
    
    # Créer la carte
    mapa = folium.Map(
        location=[centre_lat, centre_lon],
        zoom_start=zoom,
        tiles="OpenStreetMap"
    )
    
    # Compteurs
    points = 0
    lignes = 0
    polygones = 0
    
    # Extraire les features depuis les datalayers
    if 'datalayers' in umap_data:
        for layer in umap_data['datalayers']:
            layer_name = layer.get('properties', {}).get('name', 'Couche')
            layer_color = layer.get('properties', {}).get('color', 'blue')
            
            # Créer un groupe de couche
            feature_group = folium.FeatureGroup(name=layer_name)
            
            # Ajouter les features de cette couche
            if 'features' in layer:
                for feature in layer['features']:
                    geometry = feature.get("geometry", {})
                    properties = feature.get("properties", {})
                    
                    name = properties.get("name", "Point")
                    description = properties.get("description", "")
                    
                    # === Points ===
                    if geometry.get("type") == "Point":
                        coords = geometry.get("coordinates", [])
                        if coords and len(coords) >= 2:
                            popup_text = f"<b>{name}</b><br>{description}"
                            folium.Marker(
                                location=[coords[1], coords[0]],
                                popup=folium.Popup(popup_text, max_width=300),
                                tooltip=name,
                                icon=folium.Icon(color='blue', icon='info-sign')
                            ).add_to(feature_group)
                            points += 1
                    
                    # === Lignes (itinéraires) ===
                    elif geometry.get("type") == "LineString":
                        coords = [[c[1], c[0]] for c in geometry.get("coordinates", [])]
                        if coords:
                            folium.PolyLine(
                                coords,
                                color=layer_color,
                                weight=3,
                                opacity=0.7,
                                popup=name,
                                tooltip=name
                            ).add_to(feature_group)
                            lignes += 1
                    
                    # === Polygones ===
                    elif geometry.get("type") == "Polygon":
                        coords = [[[c[1], c[0]] for c in ring] 
                                 for ring in geometry.get("coordinates", [])]
                        if coords:
                            folium.Polygon(
                                locations=coords[0],
                                color=layer_color,
                                fill=True,
                                fillOpacity=0.5,
                                popup=name,
                                tooltip=name
                            ).add_to(feature_group)
                            polygones += 1
            
            # Ajouter le groupe de couche à la carte
            feature_group.add_to(mapa)
    
    # Ajouter contrôle des couches
    folium.LayerControl().add_to(mapa)
    
    # Sauvegarder
    mapa.save(output_html)
    print(f"✓ Carte générée : {output_html}")
    print(f"  - {points} points")
    print(f"  - {lignes} lignes")
    print(f"  - {polygones} polygones")
    
    return output_html

# Générer la carte
if geojson:
    html_file = generer_carte_folium(geojson)


🎨 Génération de la carte Folium...
✓ Carte générée : carte_temp.html
  - 0 points
  - 0 lignes
  - 0 polygones


In [14]:
# Afficher la carte directement dans le notebook
from IPython.display import IFrame
if geojson:
    display(IFrame(src=html_file, width=700, height=500))


In [16]:
# ⚠️ Cette étape nécessite Chrome/Chromium installé

def html_to_image_selenium(html_file, output_image="carte.png"):
    """Convertit le HTML Folium en image PNG avec Selenium"""
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        import time
        
        print("📷 Conversion HTML → PNG avec Selenium...")
        
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--start-maximized")
        options.add_argument("--disable-blink-features=AutomationControlled")
        
        driver = webdriver.Chrome(options=options)
        driver.get(f"file://{os.path.abspath(html_file)}")
        time.sleep(3)
        
        # Prendre une capture d'écran
        driver.save_screenshot(output_image)
        driver.quit()
        
        print(f"✓ Image généée : {output_image}")
        return output_image
    except Exception as e:
        print(f"⚠️  Erreur Selenium : {e}")
        print("   → Utilisez une capture d'écran manuelle à la place")
        return None

# Décommenter pour utiliser
# image_file = html_to_image_selenium(html_file, "carte.png")


In [17]:
def creer_pdf_guide(titre, description, image_carte=None, 
                    donnees_geojson=None, output_pdf="mon_guide.pdf"):
    """
    Génère un PDF professionnel avec la carte et les informations
    """
    print(f"📄 Création du PDF : {output_pdf}")
    
    doc = SimpleDocTemplate(
        output_pdf,
        pagesize=A4,
        rightMargin=1*cm,
        leftMargin=1*cm,
        topMargin=1.5*cm,
        bottomMargin=1.5*cm
    )
    
    story = []
    styles = getSampleStyleSheet()
    
    # === Style personnalisé pour le titre ===
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=24,
        textColor="#2c3e50",
        spaceAfter=20,
        alignment=TA_CENTER,
        fontName='Helvetica-Bold'
    )
    
    # === Titre ===
    story.append(Paragraph(titre, title_style))
    story.append(Spacer(1, 0.3*cm))
    
    # === Description ===
    if description:
        desc_style = ParagraphStyle(
            'Description',
            parent=styles['Normal'],
            fontSize=11,
            textColor="#555555",
            alignment=TA_CENTER,
            spaceAfter=15
        )
        story.append(Paragraph(description, desc_style))
        story.append(Spacer(1, 0.5*cm))
    
    # === Carte ===
    if image_carte and os.path.exists(image_carte):
        try:
            story.append(Paragraph("<b>Carte</b>", styles['Heading2']))
            img = RLImage(image_carte, width=17*cm, height=11*cm)
            story.append(img)
            story.append(Spacer(1, 0.8*cm))
        except Exception as e:
            print(f"⚠️  Impossible d'insérer l'image : {e}")
    
    # === Points d'intérêt ===
    if donnees_geojson and donnees_geojson.get("features"):
        story.append(Paragraph("<b>Points d'intérêt</b>", styles['Heading2']))
        story.append(Spacer(1, 0.2*cm))
        
        for idx, feature in enumerate(donnees_geojson['features'], 1):
            props = feature.get("properties", {})
            name = props.get("name", f"Point {idx}")
            desc = props.get("description", "")
            
            # Numéro et nom
            text = f"<b>{idx}. {name}</b>"
            story.append(Paragraph(text, styles['Normal']))
            
            # Description
            if desc:
                story.append(Paragraph(f"<i>{desc}</i>", styles['Normal']))
            
            story.append(Spacer(1, 0.15*cm))
    
    # === Pied de page ===
    story.append(Spacer(1, 0.5*cm))
    footer_style = ParagraphStyle(
        'Footer',
        parent=styles['Normal'],
        fontSize=9,
        textColor="#999999",
        alignment=TA_CENTER
    )
    story.append(Paragraph("Généré automatiquement depuis Umap • 2026", footer_style))
    
    # Générer le PDF
    doc.build(story)
    print(f"✓ PDF créé avec succès : {output_pdf}")
    return output_pdf

# Créer le PDF
if geojson:
    pdf_file = creer_pdf_guide(
        titre=TITRE_GUIDE,
        description=DESCRIPTION,
        image_carte=None,  # Mettre "carte.png" si vous avez l'image
        donnees_geojson=geojson,
        output_pdf=OUTPUT_PDF
    )


📄 Création du PDF : mon_guide_voyage.pdf
✓ PDF créé avec succès : mon_guide_voyage.pdf


In [22]:
from geopy.geocoders import Nominatim

# Remplacez par le nom de votre application et votre email
geolocator = Nominatim(user_agent="mon_application_berlin_v1 (aymeric.luneau@egmail.com)") 


In [25]:
import folium

# Coordonnées de Berlin (latitude, longitude)
berlin_coords = [52.5200, 13.4050]

# Créer la carte centrée sur Berlin
carte_berlin = folium.Map(location=berlin_coords, zoom_start=12)

# Ajouter un marqueur à la Porte de Brandebourg
folium.Marker(
    location=[52.5163, 13.3777],
    popup="Porte de Brandebourg, Berlin",
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(carte_berlin)

# Sauvegarder la carte dans un fichier HTML
carte_berlin.save("carte_berlin.html")
